In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q "duckdb>=0.10.0" plotly nbstripout

In [ ]:
from pathlib import Path

BASE_DIR    = Path("/content/drive/MyDrive/form13f-data")
DUCKDB_PATH = BASE_DIR / "sec13f.duckdb"

# Verify
tsvs = list(BASE_DIR.glob("*.tsv"))
print(f"✅ Found {len(tsvs)} TSV files:")
for f in tsvs:
    print(f"   {f.name}  ({f.stat().st_size/1e9:.2f} GB)")

In [ ]:
import duckdb
from pathlib import Path

# If you already defined BASE_DIR earlier, reuse it; otherwise:
BASE_DIR    = Path("/content/drive/MyDrive/form13f-data")
DUCKDB_PATH = BASE_DIR / "sec13f.duckdb"

con = duckdb.connect(str(DUCKDB_PATH))

con.execute("""
    SET memory_limit='8GB';
    SET threads=2;
    SET temp_directory='/content/tmp';
""")
print(f"✅ DuckDB connected at {DUCKDB_PATH}")

In [ ]:
from google.colab import userdata
import os, shutil

token    = userdata.get('GITHUB_TOKEN')
username = userdata.get('GITHUB_USERNAME')
REPO_DIR = Path("/content/repo")
REPO     = f"{username}/form13f-institutional-ownership"
NOTEBOOK_ON_DRIVE = BASE_DIR / "01dec2025-28feb2026_form13f.ipynb"

# Clone repo if not already done this session
if not REPO_DIR.exists():
    !git clone https://{token}@github.com/{REPO}.git {REPO_DIR}

!git config --global user.email "sudhars97@gmail.com"
!git config --global user.name "{sudhars97}"

# Copy from Drive → repo, strip outputs, commit, push
shutil.copy(str(NOTEBOOK_ON_DRIVE), str(REPO_DIR / "01_data_ingestion_overview.ipynb"))
!nbstripout {REPO_DIR}/01_data_ingestion_overview.ipynb

%cd {REPO_DIR}
!git add .
!git commit -m "notebook 01: data ingestion and macro overview"
!git push origin main
print("✅ Pushed to GitHub")

In [ ]:
files = {
    "summarypage":  BASE_DIR / "SUMMARYPAGE.tsv",
    "submission":   BASE_DIR / "SUBMISSION.tsv",
    "signature":    BASE_DIR / "SIGNATURE.tsv",
    "othermanager2": BASE_DIR / "OTHERMANAGER2.tsv",
    "othermanager":  BASE_DIR / "OTHERMANAGER.tsv",
    "infotable":    BASE_DIR / "INFOTABLE.tsv",
    "coverpage":    BASE_DIR / "COVERPAGE.tsv",
}

print("📥 Importing TSVs into DuckDB (first time only)...")
for table_name, path in files.items():
    con.execute(f"""
        CREATE OR REPLACE TABLE {table_name} AS
        SELECT * FROM read_csv_auto('{path}', delim='\t')
    """)
    print(f"✅ {table_name}")

print("✅ All base tables ready")

In [ ]:
# Clean merged_13f VIEW with only existing columns

con.execute("""
CREATE OR REPLACE VIEW merged_13f AS
SELECT
    i.ACCESSION_NUMBER                                   AS accession_number,
    s.TABLEENTRYTOTAL                                    AS table_entry_total,
    s.TABLEVALUETOTAL                                    AS table_value_total,
    b.FILING_DATE                                        AS filing_date,
    b.SUBMISSIONTYPE                                     AS submission_type,
    b.CIK                                                AS cik,
    b.PERIODOFREPORT                                     AS period_of_report,
    g.NAME                                               AS signer_name,
    g.TITLE                                              AS signer_title,
    g.SIGNATUREDATE                                      AS signature_date,
    c.FILINGMANAGER_NAME                                 AS filing_manager_name,
    c.FILINGMANAGER_CITY                                 AS filing_manager_city,
    c.FILINGMANAGER_ZIPCODE                              AS filing_manager_zipcode,
    -- position-level fields
    i.NAMEOFISSUER                                       AS nameofissuer,
    i.TITLEOFCLASS                                       AS titleofclass,
    i.CUSIP                                              AS cusip,
    i.VALUE                                              AS holding_value_thousands,
    i.SSHPRNAMT                                          AS share_qty,
    i.SSHPRNAMTTYPE                                      AS share_type,
    i.PUTCALL                                            AS put_call,
    i.INVESTMENTDISCRETION                               AS investment_discretion,
    i.VOTING_AUTH_SOLE                                   AS voting_sole,
    i.VOTING_AUTH_SHARED                                 AS voting_shared,
    i.VOTING_AUTH_NONE                                   AS voting_none
FROM infotable i
LEFT JOIN summarypage s   ON i.ACCESSION_NUMBER = s.ACCESSION_NUMBER
LEFT JOIN submission  b   ON i.ACCESSION_NUMBER = b.ACCESSION_NUMBER
LEFT JOIN signature   g   ON i.ACCESSION_NUMBER = g.ACCESSION_NUMBER
LEFT JOIN coverpage   c   ON i.ACCESSION_NUMBER = c.ACCESSION_NUMBER;
""")

print("✅ Cleaned merged_13f VIEW created successfully")

In [ ]:
#1. Macro overview of the dataset

import pandas as pd

macro_df = con.sql("""
WITH positions AS (
    SELECT
        accession_number,
        filing_manager_name,
        -- holding_value_thousands is in USD thousands
        SUM(holding_value_thousands) AS portfolio_value_thousands,
        COUNT(*)                     AS holdings_count
    FROM merged_13f
    GROUP BY 1,2
),
macro AS (
    SELECT
        COUNT(DISTINCT accession_number)          AS unique_filings,
        COUNT(*)                                  AS total_holdings,
        SUM(portfolio_value_thousands) / 1e6     AS total_aum_trillions,
        AVG(portfolio_value_thousands) * 1e3     AS avg_portfolio_usd,
        AVG(holdings_count)                      AS avg_holdings_per_filing
    FROM positions
)
SELECT * FROM macro
""").df()

macro_df

In [ ]:
#2. Portfolio size buckets

bucket_df = con.sql("""
WITH positions AS (
    SELECT
        accession_number,
        filing_manager_name,
        SUM(holding_value_thousands) AS portfolio_value_thousands,
        COUNT(*)                     AS holdings_count
    FROM merged_13f
    GROUP BY 1,2
),
with_usd AS (
    SELECT
        accession_number,
        filing_manager_name,
        portfolio_value_thousands * 1e3 AS portfolio_value_usd,
        holdings_count
    FROM positions
),
bucketed AS (
    SELECT
        *,
        CASE
            WHEN portfolio_value_usd < 1e9          THEN '< $1B'
            WHEN portfolio_value_usd < 1e10         THEN '$1-10B'
            WHEN portfolio_value_usd < 1e11         THEN '$10-100B'
            ELSE '> $100B'
        END AS portfolio_bucket
    FROM with_usd
)
SELECT
    portfolio_bucket,
    COUNT(*)                    AS filing_count,
    AVG(holdings_count)         AS avg_holdings,
    AVG(portfolio_value_usd)/1e9 AS avg_portfolio_billions
FROM bucketed
GROUP BY portfolio_bucket
ORDER BY
    CASE portfolio_bucket
        WHEN '< $1B' THEN 1
        WHEN '$1-10B' THEN 2
        WHEN '$10-100B' THEN 3
        ELSE 4
    END;
""").df()

bucket_df

In [ ]:
#3. Top managers by AUM and holdings

top_managers_df = con.sql("""
WITH positions AS (
    SELECT
        accession_number,
        filing_manager_name,
        period_of_report,
        SUM(holding_value_thousands) AS portfolio_value_thousands,
        COUNT(*)                     AS holdings_count
    FROM merged_13f
    GROUP BY 1,2,3
)
SELECT
    accession_number,
    filing_manager_name AS manager_name,
    period_of_report    AS quarter_end,
    portfolio_value_thousands * 1e3 / 1e9 AS portfolio_billions,
    holdings_count
FROM positions
ORDER BY portfolio_value_thousands DESC
LIMIT 20;
""").df()

top_managers_df

In [ ]:
plot_mgrs = top_managers_df.copy()
plot_mgrs["portfolio_billions"] = plot_mgrs["portfolio_billions"].astype(float)

fig_mgrs = px.bar(
    plot_mgrs.sort_values("portfolio_billions", ascending=False),
    x="manager_name",
    y="portfolio_billions",
    title="Top 20 managers by reported AUM (current period)",
    labels={"manager_name": "Manager", "portfolio_billions": "Portfolio size (USD billions)"}
)
fig_mgrs.update_layout(
    xaxis_tickangle=-45,
    height=500,
    template="plotly_white"
)
fig_mgrs.show()

In [ ]:
#4. Top issuers by total value

top_issuers_df = con.sql("""
SELECT
    nameofissuer,
    SUM(holding_value_thousands) * 1e3 AS total_value_usd
FROM merged_13f
GROUP BY 1
ORDER BY total_value_usd DESC
LIMIT 20;
""").df()

top_issuers_df

In [ ]:
import plotly.express as px

# Make a copy so we don't mutate the original
plot_issuers = top_issuers_df.copy()
plot_issuers["total_value_trillions"] = plot_issuers["total_value_usd"] / 1e12

fig_issuers = px.bar(
    plot_issuers.sort_values("total_value_trillions", ascending=False),
    x="nameofissuer",
    y="total_value_trillions",
    title="Top 20 issuers by total 13F-reported value",
    labels={"nameofissuer": "Issuer", "total_value_trillions": "Total value (USD trillions)"}
)
fig_issuers.update_layout(
    xaxis_tickangle=-45,
    height=500,
    template="plotly_white"
)
fig_issuers.show()

In [ ]:
#5. Top‑5 holding concentration per portfolio

concentration_df = con.sql("""
WITH positions AS (
    SELECT
        accession_number,
        filing_manager_name,
        period_of_report,
        nameofissuer,
        holding_value_thousands * 1e3 AS holding_value_usd
    FROM merged_13f
),
ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY accession_number
            ORDER BY holding_value_usd DESC
        ) AS rnk
    FROM positions
),
portfolio_totals AS (
    SELECT
        accession_number,
        SUM(holding_value_usd) AS portfolio_value_usd
    FROM positions
    GROUP BY 1
),
top5 AS (
    SELECT
        r.accession_number,
        SUM(r.holding_value_usd) AS top5_value_usd
    FROM ranked r
    WHERE r.rnk <= 5
    GROUP BY 1
),
metrics AS (
    SELECT
        p.accession_number,
        p.portfolio_value_usd,
        t.top5_value_usd,
        100.0 * t.top5_value_usd / NULLIF(p.portfolio_value_usd,0) AS top5_pct
    FROM portfolio_totals p
    JOIN top5 t USING (accession_number)
)
SELECT
    COUNT(*)                              AS portfolios_analyzed,
    AVG(top5_pct)                         AS avg_top5_concentration_pct,
    MIN(top5_pct)                         AS min_concentration_pct,
    MAX(top5_pct)                         AS max_concentration_pct
FROM metrics;
""").df()

concentration_df

In [ ]:
#6. Distribution of AUM and holdings (quantiles)

quantiles_df = con.sql("""
WITH positions AS (
    SELECT
        accession_number,
        SUM(holding_value_thousands) * 1e3 AS portfolio_value_usd,
        COUNT(*)                           AS holdings_count
    FROM merged_13f
    GROUP BY 1
),
stats AS (
    SELECT
        'AUM Quantiles ($B)' AS category,
        quantile_cont(portfolio_value_usd/1e9, 0.10) AS p10,
        quantile_cont(portfolio_value_usd/1e9, 0.25) AS p25,
        quantile_cont(portfolio_value_usd/1e9, 0.50) AS p50,
        quantile_cont(portfolio_value_usd/1e9, 0.75) AS p75,
        quantile_cont(portfolio_value_usd/1e9, 0.90) AS p90
    FROM positions
    UNION ALL
    SELECT
        'Holdings Quantiles' AS category,
        quantile_cont(holdings_count::DOUBLE, 0.10) AS p10,
        quantile_cont(holdings_count::DOUBLE, 0.25) AS p25,
        quantile_cont(holdings_count::DOUBLE, 0.50) AS p50,
        quantile_cont(holdings_count::DOUBLE, 0.75) AS p75,
        quantile_cont(holdings_count::DOUBLE, 0.90) AS p90
    FROM positions
)
SELECT * FROM stats;
""").df()

quantiles_df

In [ ]:
#7. Multi‑filing managers (aggregate across multiple filings)

multi_filing_df = con.sql("""
WITH portfolios AS (
    SELECT
        accession_number,
        filing_manager_name,
        SUM(holding_value_thousands) * 1e3 AS portfolio_value_usd
    FROM merged_13f
    GROUP BY 1,2
),
agg AS (
    SELECT
        filing_manager_name AS manager,
        COUNT(*)            AS filings,
        SUM(portfolio_value_usd)/1e9 AS total_aum_billions,
        AVG(portfolio_value_usd)/1e9 AS avg_portfolio_billions
    FROM portfolios
    GROUP BY filing_manager_name
    HAVING COUNT(*) >= 2
)
SELECT *
FROM agg
ORDER BY total_aum_billions DESC
LIMIT 20;
""").df()

multi_filing_df

In [ ]:
# Manager × Issuer heatmap (near-square N×N grid)
# ------------------------------------------------
# This cell:
# 1) Finds the top N issuers by total 13F value
# 2) For those issuers, finds the top N managers by total exposure
# 3) Builds a manager×issuer matrix of USD billions
# 4) Plots a Plotly heatmap with a near-square shape

import plotly.express as px

# Choose grid size (e.g. 20, 25, 30)
N = 30

# 1) Top N issuers by total reported value
top_issuers_dyn = con.sql(f"""
    SELECT
        nameofissuer,
        SUM(holding_value_thousands) * 1e3 AS total_value_usd
    FROM merged_13f
    GROUP BY 1
    ORDER BY total_value_usd DESC
    LIMIT {N}
""").df()

issuer_list = top_issuers_dyn["nameofissuer"].tolist()

# 2) Manager–issuer exposures for those issuers, and top N managers
placeholders = ",".join(["?"] * len(issuer_list))

mgr_issuer_df = con.execute(f"""
    WITH positions AS (
        SELECT
            filing_manager_name,
            nameofissuer,
            SUM(holding_value_thousands) * 1e3 AS total_value_usd
        FROM merged_13f
        WHERE nameofissuer IN ({placeholders})
        GROUP BY 1,2
    ),
    top_managers AS (
        SELECT
            filing_manager_name,
            SUM(total_value_usd) AS total_mgr_value_usd
        FROM positions
        GROUP BY 1
        ORDER BY total_mgr_value_usd DESC
        LIMIT {N}
    )
    SELECT
        p.filing_manager_name,
        p.nameofissuer,
        p.total_value_usd / 1e9 AS position_value_billions
    FROM positions p
    JOIN top_managers m
      ON p.filing_manager_name = m.filing_manager_name
""", issuer_list).df()

# 3) Pivot to matrix and fill missing cells with 0
heat_matrix = mgr_issuer_df.pivot_table(
    index="filing_manager_name",
    columns="nameofissuer",
    values="position_value_billions",
    fill_value=0.0
)

# 4) Plot near-square heatmap
fig = px.imshow(
    heat_matrix.values,
    labels=dict(x="Issuer", y="Manager", color="Value (USD billions)"),
    x=heat_matrix.columns,
    y=heat_matrix.index,
    title=f"Manager vs issuer holdings heatmap ({N}×{N})"
)
fig.update_layout(
    height=900,
    width=900,
    template="plotly_white"
)
fig.update_xaxes(tickangle=-45)
fig.show()

In [ ]:
# Number of managers holding each top issuer (breadth of ownership)

# 1) Take the same top issuers by value
top_issuers_dyn = con.sql("""
    SELECT
        nameofissuer,
        SUM(holding_value_thousands) * 1e3 AS total_value_usd
    FROM merged_13f
    GROUP BY 1
    ORDER BY total_value_usd DESC
    LIMIT 20
""").df()

issuer_list = top_issuers_dyn["nameofissuer"].tolist()
placeholders = ",".join(["?"] * len(issuer_list))

# 2) Count distinct managers per issuer
breadth_df = con.execute(f"""
    SELECT
        nameofissuer,
        COUNT(DISTINCT filing_manager_name) AS distinct_managers
    FROM merged_13f
    WHERE nameofissuer IN ({placeholders})
    GROUP BY 1
    ORDER BY distinct_managers DESC
""", issuer_list).df()

# 3) Plot bar chart
import plotly.express as px

fig_breadth = px.bar(
    breadth_df.sort_values("distinct_managers", ascending=False),
    x="nameofissuer",
    y="distinct_managers",
    title="Number of distinct managers holding each top issuer",
    labels={"nameofissuer": "Issuer", "distinct_managers": "# of managers"}
)
fig_breadth.update_layout(
    xaxis_tickangle=-45,
    template="plotly_white"
)
fig_breadth.show()

In [ ]:
# AUM vs holdings count (size vs diversification scatter)
# 1) Build portfolio-level dataset
aum_holdings_df = con.sql("""
WITH positions AS (
    SELECT
        accession_number,
        filing_manager_name,
        SUM(holding_value_thousands) * 1e3 AS portfolio_value_usd,
        COUNT(*)                           AS holdings_count
    FROM merged_13f
    GROUP BY 1,2
)
SELECT
    accession_number,
    filing_manager_name,
    portfolio_value_usd / 1e9 AS portfolio_billion,
    holdings_count
FROM positions
""").df()

# 2) Scatter plot (log scale on AUM)
import plotly.express as px

fig_scatter = px.scatter(
    aum_holdings_df,
    x="portfolio_billion",
    y="holdings_count",
    hover_data=["filing_manager_name"],
    title="Portfolio size vs number of holdings",
    labels={
        "portfolio_billion": "Portfolio size (USD billions, log scale)",
        "holdings_count": "Number of holdings"
    }
)
fig_scatter.update_xaxes(type="log")
fig_scatter.update_layout(template="plotly_white")
fig_scatter.show()